## **Email AI Dataset MVP Pipeline**

1. **Data Source**
   - Raw `.eml` email files

2. **Parsing**
   - Batch parse emails using Python (`email` library)

3. **Structuring**
   - Convert parsed data into structured format (JSON / CSV)

4. **Cleaning**
   - Remove HTML tags
   - Normalize text
   - Filter signatures / noise

5. **Dataset Construction**
   - Extract customer queries
   - Extract agent responses
   - Build AI-ready training pairs

            ┌────────────┐
            │ Email Input│
            └─────┬──────┘
                  ↓
            ┌────────────┐
            │ Classifier │  ← 你现在做的
            └─────┬──────┘
                  ↓
        ┌──────────────────┐
        │ Decision Engine  │
        └─────┬────────────┘
              ↓
    ┌───────────────────────┐
    │ Action                │
    │ - Auto reply          │
    │ - Route to team       │
    │ - Generate draft      │
    └───────────────────────┘

### **Configuration**

In [5]:
# 检查当前工作目录
import os
from pathlib import Path

print("cwd:", os.getcwd())

cwd: /Users/promab/anaconda_projects/email_agent/notebooks


In [6]:
# 规范化路径
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "sample_emails"
PARSED_DIR = PROJECT_ROOT / "data" / "parsed"

PARSED_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("RAW_DIR =", RAW_DIR)
print("PARSED_DIR =", PARSED_DIR)
print("files:", list(RAW_DIR.iterdir())[:5])

PROJECT_ROOT = /Users/promab/anaconda_projects/email_agent
RAW_DIR = /Users/promab/anaconda_projects/email_agent/data/raw/sample_emails
PARSED_DIR = /Users/promab/anaconda_projects/email_agent/data/parsed
files: [PosixPath('/Users/promab/anaconda_projects/email_agent/data/raw/sample_emails/New Purchase Order Number_ 4400110588.eml'), PosixPath('/Users/promab/anaconda_projects/email_agent/data/raw/sample_emails/RE_ AM06718PU-N [Ticket#2021110352000597].eml'), PosixPath('/Users/promab/anaconda_projects/email_agent/data/raw/sample_emails/Re_ Purchase order_ 4102564983.eml'), PosixPath('/Users/promab/anaconda_projects/email_agent/data/raw/sample_emails/RE_ Question_ VENDOR_011212_Replenishment _R0000005019_From 2022-04-07 at 124501.eml'), PosixPath('/Users/promab/anaconda_projects/email_agent/data/raw/sample_emails/BioLegend Customer Statement (107325).eml')]


In [7]:
from email import policy
from email.parser import BytesParser
from bs4 import BeautifulSoup
import pandas as pd

### **Email Parsing**

In [8]:
# 将html文件转换为纯文本
def clean_html_to_text(html):
    if not html:
        return None
    try:
        soup = BeautifulSoup(html, "lxml")
    except Exception:
        soup = BeautifulSoup(html, "html.parser")
    return soup.get_text(separator="\n", strip=True)

In [9]:
# 解析邮件
def parse_eml(file_path):
    file_path = Path(file_path)
    # 将.eml文件解析成一个msg对象
    with open(file_path, "rb") as f:
        msg = BytesParser(policy=policy.default).parse(f)
    
    # 提取邮件头
    subject = msg.get("subject")
    from_ = msg.get("from")
    to_ = msg.get("to")
    cc_ = msg.get("cc")
    date_ = msg.get("date")
    message_id = msg.get("message-id")
    in_reply_to = msg.get("in-reply-to")
    references = msg.get("references")

    # 初始化正文和附件
    body_text = None
    body_html = None
    attachments = []

    # 处理multipart邮件(plain text + html + attachments + inline image)
    if msg.is_multipart():
        for part in msg.walk():
            content_disposition = part.get_content_disposition()
            content_type = part.get_content_type()

            if content_disposition == "attachment":
                attachments.append(part.get_filename())
                continue

            if content_type == "text/plain" and body_text is None:
                try:
                    body_text = part.get_content()
                except Exception:
                    payload = part.get_payload(decode=True)
                    body_text = payload.decode(errors="ignore") if payload else None

            if content_type == "text/html" and body_html is None:
                try:
                    body_html = part.get_content()
                except Exception:
                    payload = part.get_payload(decode=True)
                    body_html = payload.decode(errors="ignore") if payload else None
    else:
        content_type = msg.get_content_type()

        if content_type == "text/plain":
            try:
                body_text = msg.get_content()
            except Exception:
                payload = msg.get_payload(decode=True)
                body_text = payload.decode(errors="ignore") if payload else None

        elif content_type == "text/html":
            try:
                body_html = msg.get_content()
            except Exception:
                payload = msg.get_payload(decode=True)
                body_html = payload.decode(errors="ignore") if payload else None

    if body_text is None and body_html is not None:
        body_text = clean_html_to_text(body_html)

    return {
        "file_name": file_path.name,
        "subject": subject,
        "from": from_,
        "to": to_,
        "cc": cc_,
        "date": date_,
        "message_id": message_id,
        "in_reply_to": in_reply_to,
        "references": references,
        "body_text": body_text,
        "body_html": body_html,
        "attachments": attachments,
    }

In [10]:
eml_files = sorted(RAW_DIR.glob("*.eml")) # 在raw_dir下面找所有以eml为结尾的文件 sorted即按照文件名排序
print("number of eml files:", len(eml_files))
print(eml_files[:3])

number of eml files: 20
[PosixPath('/Users/promab/anaconda_projects/email_agent/data/raw/sample_emails/BioLegend Customer Statement (107325).eml'), PosixPath('/Users/promab/anaconda_projects/email_agent/data/raw/sample_emails/New Purchase Order Number_ 4400110588.eml'), PosixPath('/Users/promab/anaconda_projects/email_agent/data/raw/sample_emails/PCOECR-0011275 - RUO TF MA517074 - 30466P - FceR1 alpha Monoclonal Antibody (1F2A9).eml')]


In [11]:
records = [] # 用于存储解析之后的结果

for fp in eml_files:
    try:
        records.append(parse_eml(fp))
    except Exception as e:
        records.append({
            "file_name": fp.name,
            "subject": None,
            "from": None,
            "to": None,
            "cc": None,
            "date": None,
            "message_id": None,
            "in_reply_to": None,
            "references": None,
            "body_text": None,
            "body_html": None,
            "attachments": None,
            "error": str(e)
        })

# 将解析结果转换为df
df = pd.DataFrame(records)

In [12]:
df

,file_name,subject,from,to,cc,date,message_id,in_reply_to,references,body_text,body_html,attachments
0,BioLegend Customer Statement (107325).eml,BioLegend Customer Statement (107325),"""Accounting Dept. "" <cs@biolegend.com>","judy.tse@promab.com, info@promab.com",NaN,"Sat, 03 Jan 2026 20:15:44 -0800",<PRODUTIL020oMBtnXYQ00011e13@PRODUTIL02.BioLeg...,NaN,NaN,"Dear Accounts Payable,\nPlease find the attach...","<meta http-equiv=""Content-Type"" content=""text/...",[107325.pdf]
1,New Purchase Order Number_ 4400110588.eml,New Purchase Order Number: 4400110588,podorders@milliporesigma.com,Judy.tse@promab.com,NaN,"Tue, 18 Nov 2025 10:59:03 +0000",<ADR50000020686953100005056BF98FD1FD0B190941A3...,NaN,NaN,Notification for: PROMAB BIOTECHNOLOGIES INC\n...,NaN,[New Purchase Order Number 4400110588.PDF]
2,PCOECR-0011275 - RUO TF MA517074 - 30466P - Fc...,PCOECR-0011275 - RUO TF MA517074 - 30466P - Fc...,"""Mairena, Carlos J."" <carlos.mairena@thermofis...","""info@promab.com"" <info@promab.com>, Judy Tse ...","""Villalobos, Wilson J."" <wilson.villalobos@the...","Thu, 04 Dec 2025 02:02:08 +0000",<SA1P222MB0677ECB2BCD4C88B41F18F9FECA6A@SA1P22...,NaN,NaN,"Dear supplier,\n\nI hope everything is going v...","<html xmlns:o=""urn:schemas-microsoft-com:offic...",[]
3,RE_ AM06718PU-N [Ticket#2021110352000597].eml,RE: AM06718PU-N [Ticket#2021110352000597],Judy Tse <judy.tse@promab.com>,Barbara Herr <bherr@origene.com>,NaN,"Thu, 04 Nov 2021 15:47:50 +0000",<SJ0PR06MB8341821AAC90D9ADB318E937F88D9@SJ0PR0...,<PH0PR22MB2891F7AC8C8754A8E7ABB90AA78D9@PH0PR2...,<BY5PR22MB1764E570D7F0FA37B4B4A61ECD8C9@BY5PR2...,"Hi Barbara,\r\n\r\nI hope all is well. Please ...","<html xmlns:v=""urn:schemas-microsoft-com:vml"" ...",[2022 Promab Antibody Catalog Price List Relea...
4,RE_ Abcam Order Confirmation_ 30994T.eml,RE: Abcam Order Confirmation: 30994T,ABC - Demand Management <Demand.Management@abc...,"""info@promab.com"" <info@promab.com>",ABC - Demand Management <Demand.Management@abc...,"Fri, 30 Jan 2026 10:39:54 +0000",<SJ0P222MB0249DE21353D2AF6A1AF108B909FA@SJ0P22...,<DS0PR06MB100370A34596F07EF32EFDC12B89EA@DS0PR...,<SJ0P222MB02494585C96592CDA034CA73909EA@SJ0P22...,"Hi Ju Yeon,\r\n\r\nThank you for confirmation....","<html xmlns:v=""urn:schemas-microsoft-com:vml"" ...",[]
5,RE_ Antibody development for ADA assay.eml,RE: Antibody development for ADA assay,Pershang Farshi <pershang.farshi@promab.com>,Judy Tse <judy.tse@promab.com>,NaN,"Tue, 03 May 2022 18:36:47 +0000",<PH0PR06MB843057BE2DB85DFE301A0EC98BC09@PH0PR0...,<SJ0PR06MB83413465FE1B7FFF7007EB88F8C09@SJ0PR0...,<PH0PR06MB8430E3B15541C1247CC38CBA8BF39@PH0PR0...,"Oh, no! Let me ask them! Thank you for letting...","<html xmlns:v=""urn:schemas-microsoft-com:vml"" ...",[]
6,RE_ Inquiry about CD19 CAR-T cells.eml,RE: Inquiry about CD19 CAR-T cells,Timothy Nguyen <timothy.nguyen@promab.com>,Maria Stella Sasso <mariastella.sasso@psioxus....,Carla Cerqueira <carla.cerqueira@psioxus.com>,"Thu, 12 Aug 2021 15:57:37 +0000",<SJ0PR06MB7648D5B7D4939F245E51DFE3ECF99@SJ0PR0...,<VI1PR06MB316550471E1565676D4883C6E8F99@VI1PR0...,<HE1PR06MB3163C1BCD60A410DDEA64A59E8F79@HE1PR0...,"Hi Maria,\r\n\r\nThanks for your message. I wi...","<html xmlns:v=""urn:schemas-microsoft-com:vml"" ...",[]
7,RE_ Order form 4500556212 PROMAB BIOTECHNOLOGI...,RE: Order form 4500556212 PROMAB BIOTECHNOLOGI...,Judy Tse <judy.tse@promab.com>,GRANGER LAURE <laure.granger@univ-lyon1.fr>,NaN,"Thu, 17 Mar 2022 16:48:17 +0000",<SJ0PR06MB8341E175CB18FFA55614934BF8129@SJ0PR0...,<ECE8349C-AA59-40A3-AFCA-1AD1E9AC75A8@univ-lyo...,<D5D6418C-71A7-4531-8FF3-DE028D1C1957@univ-lyo...,"Dear Laure,\r\n\r\nThank you for your help wit...","<html xmlns:v=""urn:schemas-microsoft-com:vml"" ...",[Inv. #20015.pdf]
8,RE_ Promab Price Increase on Antibodies Effect...,RE: Promab Price Increase on Antibodies Effect...,ProSci-Fawn <fawn@prosci-inc.com>,'Judy Tse' <judy.tse@promab.com>,NaN,"Wed, 09 Aug 2023 13:58:34 -0700",<10ac01d9cb04$43997b20$cacc7160$@prosci-inc.com>,<SJ0PR06MB

In [13]:
df['body_text']

0     Dear Accounts Payable,\nPlease find the attach...
1     Notification for: PROMAB BIOTECHNOLOGIES INC\n...
2     Dear supplier,\n\nI hope everything is going v...
3     Hi Barbara,\r\n\r\nI hope all is well. Please ...
4     Hi Ju Yeon,\r\n\r\nThank you for confirmation....
5     Oh, no! Let me ask them! Thank you for letting...
6     Hi Maria,\r\n\r\nThanks for your message. I wi...
7     Dear Laure,\r\n\r\nThank you for your help wit...
8     Hi Judy,\n\n \n\nLong time no talk. I hope all...
9     Hi Dr. Zhang,\r\n\r\nSure thing! I will have o...
10    Thank you very much, Tim, just learned somethi...
11    Thank you for confirming this information.\r\n...
12    Thank you, Judy.\r\n\r\nI am doing great.  Our...
13    Hi Scott,\n\nPlease note FedEx tracking # 7751...
14    Thank you!\n\n________________________________...
15    ULINE:  THANK YOU FOR YOUR ORDER.\n800-295-551...
16    Dear Valued Supplier,\r\n\r\n\r\nWe truly appr...
17    Dear ProMab Biotechnologies, Inc.,\nEnclos

In [14]:
# 检查error行
# df[df["body_text"].isna()][["file_name", "subject", "error"]]

### **Cleaning Body Text**
1. 转义字符
2. 邮件签名和分割线等
3. url和html残留

In [15]:
import re

def clean_email_text(text):
    if text is None:
        return None

    # 1. 统一换行（关键）
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # 2. 把字面字符串 "\n" 转成真实换行
    text = text.replace("\\n", "\n")

    # 3. 去 URL
    text = re.sub(r"<https?://[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # 4. 去 HTML tag
    text = re.sub(r"</?[^>\n]+>", " ", text)

    # 5. 去分割线
    text = re.sub(r"[_-]{5,}", " ", text)

    # 6. 去多余空格
    text = re.sub(r"[ \t]+", " ", text)

    # 7. 压缩空行
    text = re.sub(r"\n\s*\n+", "\n\n", text)

    # 8. strip 每行
    text = "\n".join(line.strip() for line in text.split("\n"))

    return text.strip()

In [16]:
df["clean_text"] = df["body_text"].apply(clean_email_text)

In [17]:
# print(df["clean_text"].iloc[3])

In [18]:
df[["clean_text"]].head()

,clean_text
0,"Dear Accounts Payable,\nPlease find the attach..."
1,Notification for: PROMAB BIOTECHNOLOGIES INC\n...
2,"Dear supplier,\n\nI hope everything is going v..."
3,"Hi Barbara,\n\nI hope all is well. Please find..."
4,"Hi Ju Yeon,\n\nThank you for confirmation.\n\n..."


### **Labelling & Classification**
1. Inquiry（询单 / 产品咨询）
2. Quote Request（报价请求）
3. Order Issue（订单问题）
4. Payment Issue（付款问题）
5. Shipping / Tracking（物流问题）
6. Technical Question（技术问题）
7. Complaint（投诉）
8. Follow-up（跟进）
9. Spam / Irrelevant（无关邮件）

*仅作为初始版本的taxonomy*

### **Baseline Agent**

In [1]:
from openai import OpenAI

client = OpenAI()

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [20]:
'''
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "hello"}]
)

print(response.choices[0].message.content)
'''

'\nresponse = client.chat.completions.create(\n    model="gpt-4o-mini",\n    messages=[{"role": "user", "content": "hello"}]\n)\n\nprint(response.choices[0].message.content)\n'

In [21]:
VALID_CATEGORIES = [
    "Inquiry",
    "Quote Request",
    "Order Issue",
    "Payment Issue",
    "Shipping / Tracking",
    "Technical Question",
    "Complaint",
    "Follow-up",
    "Spam / Irrelevant"
]

In [22]:
def normalize_category(raw_category):
    raw = raw_category.strip().lower()
    for cat in VALID_CATEGORIES:
        if cat.lower() in raw:
            return cat
    return "Inquiry"

In [3]:
def classify_email(email_text):
    prompt = f"""
You are an email classification assistant.

Classify the following email into ONE of these categories only:

1. Inquiry — general product or service inquiry
2. Quote Request — asking for price, quotation, cost, or formal quote
3. Order Issue — issue with an existing order, order status, missing items, wrong items
4. Payment Issue — payment failure, invoice issue, billing question
5. Shipping / Tracking — shipment status, tracking number, delivery timing
6. Technical Question — technical details, specifications, protocol, usage question
7. Complaint — dissatisfaction, frustration, negative feedback
8. Follow-up — checking back on a previous email or pending request
9. Spam / Irrelevant — unrelated or unsolicited message

Email:
{email_text}

Return only the category name, exactly as listed above.
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a precise email classifier."},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    raw_category = response.choices[0].message.content.strip()
    return normalize_category(raw_category)

In [23]:
'''
sample_email = df["clean_text"].iloc[0]
print(sample_email)
'''

'\nsample_email = df["clean_text"].iloc[0]\nprint(sample_email)\n'

In [24]:
'''
category = classify_email(sample_email)
print(category)
'''

'\ncategory = classify_email(sample_email)\nprint(category)\n'

#### handler & routing

In [30]:
def handle_inquiry(email_text):
    return "Thanks for your inquiry. Could you please provide more details about the product you are interested in?"

def handle_quote(email_text):
    return "Thank you for your quote request. Please confirm the product name, catalog number, quantity, and shipping destination."

def handle_order_issue(email_text):
    return "We are sorry for the inconvenience. Please provide your order number so we can check the issue."

def handle_payment_issue(email_text):
    return "Thank you for reaching out regarding payment. Please share the invoice number or payment reference for further checking."

def handle_shipping(email_text):
    return "Thank you for your message. Please provide your order number or tracking number so we can check the shipment status."

def handle_technical(email_text):
    return "Thank you for your technical question. Our team will review the details and get back to you shortly."

def handle_complaint(email_text):
    return "We are sorry to hear about your experience. Your message will be escalated to our support team for immediate review."

def handle_followup(email_text):
    return "Thank you for your follow-up. We will review the previous conversation and respond as soon as possible."

def handle_spam(email_text):
    return "This email has been marked as irrelevant."

def handle_default(email_text):
    return "Thank you for your email. Our team will get back to you soon."

In [31]:
def route_email(category, email_text):
    if category == "Inquiry":
        return handle_inquiry(email_text)
    elif category == "Quote Request":
        return handle_quote(email_text)
    elif category == "Order Issue":
        return handle_order_issue(email_text)
    elif category == "Payment Issue":
        return handle_payment_issue(email_text)
    elif category == "Shipping / Tracking":
        return handle_shipping(email_text)
    elif category == "Technical Question":
        return handle_technical(email_text)
    elif category == "Complaint":
        return handle_complaint(email_text)
    elif category == "Follow-up":
        return handle_followup(email_text)
    elif category == "Spam / Irrelevant":
        return handle_spam(email_text)
    else:
        return handle_default(email_text)

#### agent

In [32]:
def email_agent(email_text):
    category = classify_email(email_text)
    reply = route_email(category, email_text)

    return {
        "category": category,
        "reply": reply
    }

In [33]:
sample_email = df["clean_text"].iloc[0]
print(sample_email)

Dear Accounts Payable,
Please find the attached statement showing open invoices ($ 617.66). We want to confirm you have received these invoices and are currently processing them for payment.
Thank you for your cooperation.
Accounting Department
BioLegend Inc - 8999 BioLegend Way, San Diego CA 92121
Phone (US & Canada): (858) 455-9588
Fax: (858) 455-9587
This is a system generated mail.


In [34]:
result = email_agent(sample_email)
print(result)

{'category': 'Payment Issue', 'reply': 'Thank you for reaching out regarding payment. Please share the invoice number or payment reference for further checking.'}


#### quote从固定回复到引用llm升级

In [36]:
def handle_quote(email_text):

    prompt = f"""
You are a professional customer support agent at a biotechnology company.

A customer sent the following email:

{email_text}

Write a polite and professional reply asking for necessary details to provide a quotation.
Mention:
- product name
- catalog number
- quantity
- shipping destination

Keep it concise and professional.
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )

    return response.choices[0].message.content.strip()

In [38]:
result = email_agent(sample_email)
print(result)

{'category': 'Payment Issue', 'reply': 'Thank you for reaching out regarding payment. Please share the invoice number or payment reference for further checking.'}
